# Binary Default Baseline

## Objective
This notebook evaluates a binary baseline for default prediction.

We train a neural network with the target defined as:

- `1 = default`
- `0 = continue or prepay`

The purpose of this experiment is diagnostic. We want to test whether collapsing the problem into a binary classification task hurts performance relative to the competing-risk formulation.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

MONTHLY_PATH = "/content/drive/MyDrive/loan_project_ncr/monthly_df_tb3ms_borrower.parquet"

df = pd.read_parquet(MONTHLY_PATH)

print(df.shape)
df.head()

(1656440, 35)


,loan_id,cohort_year,month_since_orig,status_next,tbill_level,tbill_change_3m,apr_tbill_spread,apr_tbill_spread_change_3m,apr_tbill_spread_pct,borrower_apr,...,credit_score_upper_is_missing,dti_is_missing,delinq_7y_is_missing,open_credit_lines_is_missing,revolving_balance_is_missing,bankcard_util_is_missing,available_bankcard_credit_is_missing,income_range_is_missing,employment_status_is_missing,homeowner_is_missing
0,0,2007,1,0.0,0.0420,-0.0053,0.1160,-0.0053,0.734177,0.158,...,0,0,0,0,0,0,0,0,0,0
1,0,2007,2,0.0,0.0389,-0.0072,0.1191,-0.0072,0.753797,0.158,...,0,0,0,0,0,0,0,0,0,0
2,0,2007,3,0.0,0.0390,-0.0092,0.1190,-0.0092,0.753165,0.158,...,0,0,0,0,0,0,0,0,0,0
3,0,2007,4,0.0,0.0327,-0.0093,0.1253,-0.0093,0.793038,0.158,...,0,0,0,0,0,0,0,0,0,0
4,0,2007,5,0.0,0.0300,-0.0089,0.1280,-0.0089,0.810127,0.158,...,0,0,0,0,0,0,0,0,0,0


## Dataset

We use the loan-month dataset built from Prosper loans and TB3MS-based macro features.

Each row corresponds to a loan-month observation. The target `status_next` is encoded as:

- `0 = continue`
- `1 = prepay`
- `2 = default`

For this notebook, we define:


In [ ]:
df["target_default"] = (df["status_next"] == 2).astype(int)

df["target_default"].value_counts()

,count
target_default,
0,1649553
1,6887


## Feature Set

We use the same feature set as in the competing-risk experiments so that the comparison isolates the effect of the modeling objective rather than the inputs.

In [ ]:
features = [
    "month_since_orig",
    "tbill_level",
    "tbill_change_3m",
    "apr_tbill_spread",
    "apr_tbill_spread_change_3m",
    "apr_tbill_spread_pct",
    "borrower_apr",
    "loan_amount_log",
    "prosper_score",
    "credit_score_lower",
    "credit_score_upper",
    "dti",
    "delinq_7y",
    "open_credit_lines",
    "revolving_balance",
    "bankcard_util",
    "available_bankcard_credit",
    "income_range",
    "employment_status",
    "homeowner",
]

In [ ]:
train_df = df[df["cohort_year"] <= 2011]
val_df = df[df["cohort_year"] == 2012]
test_df = df[df["cohort_year"].isin([2013, 2014])]

print(len(train_df), len(val_df), len(test_df))

702485 283842 670113


## Preprocessing

To avoid leakage, all preprocessing is fit on the training set only.

We apply:

- median imputation for missing values
- standardization using `StandardScaler`

The validation set is transformed using the train-fitted preprocessing objects.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_train = train_df[features]
X_val = val_df[features]

y_train = train_df["target_default"]
y_val = val_df["target_default"]

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

## Model

We use a simple feedforward neural network as a binary baseline.

Architecture:
- Linear -> ReLU -> Dropout
- Linear -> ReLU -> Dropout
- Linear output layer

Loss:
- `BCEWithLogitsLoss`

Optimizer:
- Adam

In [ ]:
import torch
import torch.nn as nn

class DefaultMLP(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim,256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128,1)
        )

    def forward(self,x):
        return self.net(x)

In [ ]:
import torch.optim as optim
from sklearn.metrics import average_precision_score

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DefaultMLP(X_train.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

## Training Procedure

The model is trained for 15 epochs and evaluated on the validation set after each epoch using PR-AUC.

PR-AUC is the main metric because the default class is highly imbalanced.

## Evaluation Metric

We report **PR-AUC** on the validation set.

This is more informative than accuracy in this setting because default events are rare.

In [ ]:
X_train_t = torch.tensor(X_train).float().to(device)
y_train_t = torch.tensor(y_train.values).float().to(device)

X_val_t = torch.tensor(X_val).float().to(device)
y_val_t = torch.tensor(y_val.values).float().to(device)

for epoch in range(15):

    model.train()

    optimizer.zero_grad()

    logits = model(X_train_t).squeeze()

    loss = criterion(logits, y_train_t)

    loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        val_logits = model(X_val_t).squeeze()

        val_probs = torch.sigmoid(val_logits).cpu().numpy()

        pr_auc = average_precision_score(y_val, val_probs)

    print(epoch, loss.item(), pr_auc)

0 0.685668408870697 0.004467967017534133
1 0.6593395471572876 0.0043425398927726755
2 0.6338047385215759 0.004271646614894189
3 0.6089616417884827 0.004245364038497586
4 0.5849284529685974 0.004278803876762921
5 0.5615333914756775 0.004291665757289481
6 0.5388002991676331 0.004324702013497838
7 0.5167774558067322 0.004358843219064152
8 0.4953756034374237 0.004391794792596818
9 0.4745499789714813 0.004414510686494872
10 0.4543057978153229 0.004414682344283389
11 0.43461835384368896 0.004426079341444759
12 0.41553646326065063 0.004438794604824676
13 0.3969588279724121 0.004446378836208765
14 0.3789716958999634 0.0044514714770239865


## Results

Validation PR-AUC increases only slightly during training and remains low.

Final validation PR-AUC is approximately **0.0045**.

## Interpretation

This binary baseline performs poorly.

Compared with the competing-risk model, the binary formulation appears to lose important structure in the problem. In particular, prepayment acts as a competing event for default, and collapsing everything into a single binary target makes the prediction task less informative.

## Conclusion

The binary default experiment suggests that a simple default-vs-not formulation is not sufficient.

This supports keeping the competing-risk setup in later experiments.